In [1]:
import os
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from PIL import Image
import requests
from io import BytesIO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
# Preprocessing & Data Augmentation: models expect images of shape (3 x H x W) normalized with ImageNet mean/std
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [3]:
# Load Datasets using ImageFolder
data_dir = "./exotic"
train_dataset = datasets.ImageFolder(root=os.path.join(data_dir, "train"), transform=train_transforms)
test_dataset = datasets.ImageFolder(root=os.path.join(data_dir, "test"), transform=test_transforms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
class_names = train_dataset.classes
num_classes = len(class_names)
print(f"Classes: {class_names} (Total: {num_classes})")

Classes: ['bear', 'chimp', 'giraffe', 'gorilla', 'llama', 'ostrich', 'porcupine', 'skunk', 'triceratops', 'zebra'] (Total: 10)


In [4]:
# Loading weights using modern torchvision Weights enum API
model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

In [5]:
# Classifier Head for multi-class prediction must match exotic classes
num_features = model.classifier[6].in_features
model.classifier[6] = nn.Linear(num_features, num_classes)
model = model.to(device)

In [6]:
# Assign Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

In [7]:
# 8. Training & Testing Loop function
def train_model(model, train_loader, test_loader, criterion, optimizer, epochs=5):
    results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    
    for epoch in range(epochs):
        model.train()
        train_loss, train_correct = 0.0, 0
        total_train = 0
        
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X.size(0)
            _, preds = torch.max(outputs, 1)
            train_correct += torch.sum(preds == y.data).item()
            total_train += X.size(0)
            
        epoch_train_loss = train_loss / total_train
        epoch_train_acc = train_correct / total_train
        model.eval()
        test_loss, test_correct = 0.0, 0
        total_val = 0
        
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                outputs = model(X)
                loss = criterion(outputs, y)
                test_loss += loss.item() * X.size(0)
                _, preds = torch.max(outputs, 1)
                test_correct += torch.sum(preds == y.data).item()
                total_val += X.size(0)
                
        epoch_test_loss = test_loss / total_val
        epoch_test_acc = test_correct / total_val
        results["train_loss"].append(epoch_train_loss)
        results["train_acc"].append(epoch_train_acc)
        results["test_loss"].append(epoch_test_loss)
        results["test_acc"].append(epoch_test_acc)
        
        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | "
              f"Test Loss: {epoch_test_loss:.4f} Acc: {epoch_test_acc:.4f}")
              
    return results

In [8]:
# Train and Save Model Weights
epochs = 5
results = train_model(model, train_loader, test_loader, criterion, optimizer, epochs=epochs)
torch.save(model.state_dict(), "vgg16_fine_tuned.pth")
print("Weights saved to vgg16_fine_tuned.pth")

Epoch 1/5 | Train Loss: 2.1660 Acc: 0.2133 | Test Loss: 1.7335 Acc: 0.6400
Epoch 2/5 | Train Loss: 1.4622 Acc: 0.5833 | Test Loss: 0.9396 Acc: 0.8600
Epoch 3/5 | Train Loss: 0.7364 Acc: 0.7833 | Test Loss: 0.4021 Acc: 0.9200
Epoch 4/5 | Train Loss: 0.3716 Acc: 0.8850 | Test Loss: 0.2571 Acc: 0.9400
Epoch 5/5 | Train Loss: 0.2472 Acc: 0.9250 | Test Loss: 0.2229 Acc: 0.9300
Weights saved to vgg16_fine_tuned.pth


In [9]:
# Data Augmentation for Predictions
predict_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [10]:
# Model Loading
def load_trained_model(weights_path, num_classes):
    model = models.vgg16(weights=None)
    num_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(num_features, num_classes)
    state_dict = torch.load(weights_path, map_location=device)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    return model

In [11]:
# Image Prediction Function
def predict_image(image_source, model, class_names):
    if os.path.exists(image_source):
        img = Image.open(image_source).convert("RGB")
        print(f"Loaded local image: {image_source}")
    else:
        print(f"File not found: {image_source}")
        return None

    img_transformed = predict_transforms(img)
    img_batch = img_transformed.unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(img_batch)
        probabilities = torch.softmax(logits, dim=1)[0]
        predicted_idx = torch.argmax(probabilities).item()
        
    predicted_class = class_names[predicted_idx]
    confidence_score = probabilities[predicted_idx].item()
    print(f"\n--- PREDICTION RESULTS ---")
    print(f"Predicted Class: {predicted_class}")
    print(f"Confidence: {confidence_score:.2%}")
    print(f"All Probabilities: { {class_names[i]: float(prob) for i, prob in enumerate(probabilities)} }")
    return predicted_class, confidence_score

In [17]:
print("Current Working Directory:", os.getcwd())
print("Contents of current directory:", os.listdir("."))
print(os.listdir("./exotic/valid/llama"))
from pathlib import Path
llama_images = list(Path("./exotic/valid/llama").glob("*.*"))

Current Working Directory: C:\Users\jayog\Documents\INFO4000\3rdexercise\src\3rdexercise
Contents of current directory: ['.gitignore', '.ipynb_checkpoints', 'exotic', 'featureExtractor.ipynb', 'fineTuner.ipynb', 'vgg16_feature_extraction.pth', 'vgg16_feature_extraction.py', 'vgg16_fine_tuned.pth', 'vgg16_fine_tuning.py', '__init__.py']
['134_0061.jpg', '134_0062.jpg', '134_0063.jpg', '134_0064.jpg', '134_0065.jpg', '134_0066.jpg', '134_0067.jpg', '134_0068.jpg', '134_0069.jpg', '134_0070.jpg']


In [18]:
if __name__ == "__main__":
    exotic_classes = ["bear", "chimp", "giraffe", "gorilla", "llama", "ostrich", "porcupine", "skunk", "triceratops", "zebra"]  # Replace with your actual Exotic class names
    num_classes = len(exotic_classes)
    weights_path = "vgg16_fine_tuned.pth"
    
    if os.path.exists(weights_path):
        trained_model = load_trained_model(weights_path, num_classes)
        if llama_images:
            sample_url = str(llama_images[0])
            print(f"Targeting image: {sample_url}")
            predict_image(sample_url, trained_model, exotic_classes)
        else:
            print("No image found in ./exotic/valid/llama")
    else:
        print(f"Weights file '{weights_path}' not found. Run model training first!")

Targeting image: exotic\valid\llama\134_0061.jpg
Loaded local image: exotic\valid\llama\134_0061.jpg

--- PREDICTION RESULTS ---
Predicted Class: llama
Confidence: 99.61%
All Probabilities: {'bear': 0.00011878420627908781, 'chimp': 3.6529138014884666e-05, 'giraffe': 0.0009493406978435814, 'gorilla': 7.823080522939563e-06, 'llama': 0.9960806965827942, 'ostrich': 0.0026963246054947376, 'porcupine': 9.504127774562221e-06, 'skunk': 2.2823232939117588e-05, 'triceratops': 2.9759359676972963e-05, 'zebra': 4.8481520934728906e-05}
